# TAC Python API Demo

Covers the full `tachiom.Tac` interface:
- **Run** Token-Aware Clustering on raw `.npy` files
- **Inspect** centroids and assignments
- **Analyse** the centroid budget allocation across token types
- **Save** centroids and assignments to disk
- **Feed** into `Tachiom.build_from_tac()` to build a full retrieval index

Input files (LOTTE, same as `tachiom_demo.ipynb`):

| File | Shape | dtype | Description |
|---|---|---|---|
| `documents.npy` | `[N, dim]` | `f16` | Token vectors |
| `document_token_ids_flat.npy` | `[N]` | `i64`/`u32` | Token-type ID per token |
| `doclens.npy` | `[n_docs]` | `i32`/`i64` | Tokens per document (only for Tachiom build) |

---
## 0b — Build the shared library

Run once per code change. `target-cpu=native` enables SIMD (AVX2/AVX-512) for PQ kernels.

In [1]:
import subprocess, sys, os

result = subprocess.run(
    ["maturin", "develop", "--release"],
    cwd="..",
    env={**os.environ, "RUSTFLAGS": "-C target-cpu=native"},
    capture_output=True,
    text=True,
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print(result.stderr[-2000:])
    raise RuntimeError("maturin build failed")
print("Build OK")

✏️ Setting installed package as editable

Build OK


---
## 1 — Configuration

In [2]:
from pathlib import Path

# ── Input paths (same dataset as tachiom_demo) ────────────────────────────────
DATA_DIR = Path("/data2/cosimorulli/real_new_lotte/lotte_new")

VECTORS_FILE   = DATA_DIR / "documents.npy"
TOKEN_IDS_FILE = DATA_DIR / "document_token_ids_flat.npy"
DOCLENS_FILE   = DATA_DIR / "doclens.npy"   # only needed for Tachiom.build_from_tac()

# ── TAC output paths ──────────────────────────────────────────────────────────
TAC_DIR         = Path("/data3/silvio/tac_output/lotte_2M")
CENTROIDS_OUT   = TAC_DIR / "centroids.npy"     # [K, dim] f32
ASSIGNMENTS_OUT = TAC_DIR / "assignments.npy"   # [N] u32

# ── TAC params ────────────────────────────────────────────────────────────────
TAC_PARAMS = dict(
    n_centroids    = 262_144,
    n_iter         = 10,
    verbose        = True,
    max_sample_size = None,   # None = auto formula
)

print("Config OK")

Config OK


---
## 2 — Run TAC

`Tac(n_centroids, *, n_iter, verbose, max_sample_size)` — configure once.  
`tac.train(vectors_path, token_ids_path)` — runs Token-Aware Clustering and populates `centroids`, `assignments`, etc.

In [3]:
import time, tachiom

tac = tachiom.Tac(**TAC_PARAMS)
print(tac)   # not yet trained

t0 = time.perf_counter()
tac.train(str(VECTORS_FILE), str(TOKEN_IDS_FILE))
print(f"\nTAC done in {time.perf_counter() - t0:.2f}s")
print(tac)   # trained

<Tac: budget=262144, not yet trained>
=== TAC: 266205513 vectors × dim=128, 28865 unique token types, budget=262144 centroids ===

=== Damped Spread Centroid Allocation ===
Total vectors: 266205513
Budget: 262144 centroids
Thresholds: Micro < 128, Small < 256
Bounds: Floor = 4, Min points/centroid = 39

--- Phase 1: Tail Handling ---
Micro tokens (< 128): 7713 tokens → 7713 centroids
Small tokens (128-256): 2943 tokens → 5886 centroids
Active tokens (≥ 256): 18209 tokens
Tail budget used: 13599
Remaining budget for active tokens: 248545

--- Phase 2: Damped Scoring ---
Computing spread measures for 18209 active tokens...
✓ Spread computation in 7.17s

Top 10 damped scores:
  1. Token 1996: count=12696959, spread=0.7982, score=2844.3571
  2. Token 2000: count=6621496, spread=0.7724, score=1987.6021
  3. Token 1037: count=5772757, spread=0.8081, score=1941.6258
  4. Token 1997: count=4260873, spread=0.7939, score=1638.6629
  5. Token 1998: count=4287062, spread=0.7818, score=1618.7100
  

---
## 3 — Inspect results

After `train()`, four properties are available: `n_centroids`, `dim`, `centroids` (f32), `centroids_f16` (f16), `assignments` (u32).

In [4]:
import numpy as np

print(f"n_centroids : {tac.n_centroids:,}")
print(f"dim         : {tac.dim}")
print()
print(f"centroids      : shape={tac.centroids.shape}    dtype={tac.centroids.dtype}")
print(f"centroids_f16  : shape={tac.centroids_f16.shape}  dtype={tac.centroids_f16.dtype}")
print(f"assignments    : shape={tac.assignments.shape}  dtype={tac.assignments.dtype}")
print()

c = tac.centroids
print(f"centroid norms  — mean={np.linalg.norm(c, axis=1).mean():.4f}  "
      f"min={np.linalg.norm(c, axis=1).min():.4f}  max={np.linalg.norm(c, axis=1).max():.4f}")
print(f"assignments     — min={tac.assignments.min()}  max={tac.assignments.max()}  "
      f"(expected max = n_centroids - 1 = {tac.n_centroids - 1})")

n_centroids : 262,144
dim         : 128

centroids      : shape=(262144, 128)    dtype=float32
centroids_f16  : shape=(262144, 128)  dtype=float16
assignments    : shape=(266205513,)  dtype=uint32

centroid norms  — mean=0.8787  min=0.2757  max=1.0001
assignments     — min=0  max=262143  (expected max = n_centroids - 1 = 262143)


In [5]:
# f32 vs f16 round-trip error
c_f32 = tac.centroids
c_f16_as_f32 = tac.centroids_f16.astype(np.float32)

abs_diff = np.abs(c_f32 - c_f16_as_f32)
print("f32 vs f16 centroid values:")
print(f"  max absolute diff  : {abs_diff.max():.2e}")
print(f"  mean absolute diff : {abs_diff.mean():.2e}")

f32 vs f16 centroid values:
  max absolute diff  : 0.00e+00
  mean absolute diff : 0.00e+00


---
## 4 — Centroid budget analysis

TAC runs separate k-means per token type and distributes the budget with a damped-spread strategy:
- **Micro** tokens (< 128 occurrences) → 1 centroid each
- **Small** tokens (128–256 occurrences) → 2 centroids each
- **Active** tokens (≥ 256 occurrences) → damped score `√count × spread`, floor=4, budget-reconciled

Here we verify those properties by recovering the allocation from `assignments` + `token_ids`.

In [6]:
# Load token IDs (same file that was passed to tac.train)
# Shape: [N]  dtype: i64  —  values are vocabulary IDs (BERT-style: 0..30521)
token_ids = np.load(TOKEN_IDS_FILE)
print(f"token_ids : shape={token_ids.shape}  dtype={token_ids.dtype}")
print(f"vocab size (unique token types) : {np.unique(token_ids).size:,}")
print(f"total tokens                    : {len(token_ids):,}")

token_ids : shape=(266205513,)  dtype=int64
vocab size (unique token types) : 28,865
total tokens                    : 266,205,513


In [7]:
# For each unique token type: count how many tokens it has (frequency)
# and how many unique centroid IDs are assigned to it (allocated centroids).
#
# Strategy: sort by token_id once, then iterate over contiguous groups.
assignments = tac.assignments

sort_idx      = np.argsort(token_ids, kind='stable')
sorted_tids   = token_ids[sort_idx]
sorted_assigns = assignments[sort_idx]

unique_tids, tok_counts = np.unique(sorted_tids, return_counts=True)
split_points = np.cumsum(tok_counts)[:-1]
groups = np.split(sorted_assigns, split_points)

n_centroids_per_tid = np.array([np.unique(g).size for g in groups])

print(f"Unique token types : {len(unique_tids):,}")
print()

# ── Token frequency distribution ─────────────────────────────────────────────
micro = tok_counts < 128
small = (tok_counts >= 128) & (tok_counts < 256)
active = tok_counts >= 256
print(f"Micro  (< 128 occ)   : {micro.sum():5,} token types")
print(f"Small  (128-255 occ) : {small.sum():5,} token types")
print(f"Active (≥ 256 occ)   : {active.sum():5,} token types")
print()

# ── Centroid allocation distribution ─────────────────────────────────────────
print("Centroids per token type:")
print(f"  min    : {n_centroids_per_tid.min()}")
print(f"  max    : {n_centroids_per_tid.max():,}")
print(f"  mean   : {n_centroids_per_tid.mean():.1f}")
print(f"  median : {int(np.median(n_centroids_per_tid))}")
print(f"  total  : {n_centroids_per_tid.sum():,}  (should equal n_centroids={tac.n_centroids:,})")
print()

# Verify micro/small floors
assert (n_centroids_per_tid[micro] == 1).all(),  "micro tokens should have exactly 1 centroid"
assert (n_centroids_per_tid[small] == 2).all(),  "small tokens should have exactly 2 centroids"
assert (n_centroids_per_tid[active] >= 4).all(), "active tokens should have at least 4 centroids"
print("✓ Micro/small floors verified")

Unique token types : 28,865

Micro  (< 128 occ)   : 7,713 token types
Small  (128-255 occ) : 2,943 token types
Active (≥ 256 occ)   : 18,209 token types

Centroids per token type:
  min    : 1
  max    : 1,958
  mean   : 9.1
  median : 4
  total  : 262,144  (should equal n_centroids=262,144)

✓ Micro/small floors verified


---
## 5 — Save to disk

Save `centroids` (f32) and `assignments` (u32) in NumPy format so they can be passed to `Tachiom.build_from_tac()` later — no need to re-run TAC.

In [10]:
TAC_DIR.mkdir(parents=True, exist_ok=True)

np.save(CENTROIDS_OUT,   tac.centroids)     # f32  [K, dim]
np.save(ASSIGNMENTS_OUT, tac.assignments)   # u32  [N]

print(f"Saved centroids   → {CENTROIDS_OUT}   ({CENTROIDS_OUT.stat().st_size / 1e9:.2f} GB)")
print(f"Saved assignments → {ASSIGNMENTS_OUT}   ({ASSIGNMENTS_OUT.stat().st_size / 1e9:.2f} GB)")

# Verify round-trip
c_rt = np.load(CENTROIDS_OUT)
a_rt = np.load(ASSIGNMENTS_OUT)
assert np.array_equal(c_rt, tac.centroids),    "centroids round-trip mismatch"
assert np.array_equal(a_rt, tac.assignments),  "assignments round-trip mismatch"
print("✓ Round-trip verified")

Saved centroids   → /data3/silvio/tac_output/lotte_2M/centroids.npy   (0.13 GB)
Saved assignments → /data3/silvio/tac_output/lotte_2M/assignments.npy   (1.06 GB)
✓ Round-trip verified


---
## 6 — Build a Tachiom index from the TAC output

`Tachiom.build_from_tac()` skips the clustering step and uses the centroids and assignments saved above.  
See `tachiom_demo.ipynb` (Option B) for full search and evaluation.

In [11]:
import time

t0 = time.perf_counter()
idx = tachiom.Tachiom.build_from_tac(
    str(VECTORS_FILE),
    str(TOKEN_IDS_FILE),
    str(DOCLENS_FILE),
    str(CENTROIDS_OUT),
    str(ASSIGNMENTS_OUT),
    pq_sample_size  = 10_000_000,
    pq_n_iter       = 10,
    normalize       = True,
    pq_seed         = 42,
    hnsw_m          = 32,
    ef_construction = 1500,
)
print(f"Built in {time.perf_counter() - t0:.2f}s")
print(idx)
idx.print_space_usage()

[Tachiom::build_index_from_tac] 2428854 docs, 266205513 tokens, dim=128, 262144 centroids
[Tachiom::build_index] Step 2: Selecting PQ training sample...
[Tachiom::build_index] PQ training sample: 9950370 tokens
[Tachiom::build_index] Step 3: Training encoder...
train_from_coarse: 9950370 tokens × dim=128, 262144 coarse centroids, M=32, dsub=4, sample=9950370, normalize=true
  Step 1: computing 9950370 training residuals...
  Step 2: training PQ (32 subspaces × 256 centroids, 10 iters)...
Running K-Means for 32 subspaces
K-Means finished
  train_from_coarse complete.
[Tachiom::build_index] Step 3: Encoding 2428854 documents...
[Tachiom::build_index] Step 4: Building HNSW on 262144 centroids...
[Tachiom::build_index] Step 5: Building inverted lists...
Built in 559.23s
<Tachiom: 2428854 docs, dim=128, 262144 centroids>
Index space usage:
  centroids_hnsw         0.10 GB  (  0.9%)
  inverted_lists         1.30 GB  ( 11.8%)
  offsets                0.00 GB  (  0.0%)
  residuals             